# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Lab 9 Data Visualization
# Name: Aleena Zahra
# Roll No: 23i-2514
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [1]:
!pip install folium


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [49]:

import folium
from folium.plugins import MarkerCluster
import pandas as pd
import requests



## Task 1 – Basic Map Rendering


In [5]:

fastMap = folium.Map(location=[33.6565, 73.0154], zoom_start=12, tiles='OpenStreetMap')
folium.Marker([33.6565, 73.0154], popup="FAST unii").add_to(fastMap)


In [6]:

fastMap.save("task1.html")

## Task 2:  Multiple Locations

In [13]:


data = {
    'Place': ['Hospital', 'Park', 'School', 'Mall', 'Restaurant'],
    'Lat': [33.6565, 33.6665, 33.6765, 33.6865, 33.6965],
    'Lon': [73.0154, 73.0254, 73.0354, 73.0454, 73.0554]
}


In [15]:



df = pd.DataFrame(data)
map2 = folium.Map(location=[33.6565,73.0154], zoom_start=12)


In [20]:


for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row['Lat'], row['Lon']],
        radius=6,
        color='red',
        fill=True,
        fill_color='orange',
        popup=row['Place']
    ).add_to(map2)


In [21]:

map2.save("task2.html")


## Task 3: Chloropleth Map

In [ ]:
!pip install geopandas

In [28]:
import geopandas as gpd

In [ ]:

geojson_path = "Pakistan_ADM1_simplified.simplified.geojson"
gdf = gpd.read_file(geojson_path)


print(gdf.head())



            shapeName shapeISO                  shapeID shapeGroup shapeType  \
0         Balochistan    PK-BA  70912109B90439427603182        PAK      ADM1   
1               Sindh    PK-SD   70912109B8393927546319        PAK      ADM1   
2    Gilgit-Baltistan    PK-GB  70912109B54485417915534        PAK      ADM1   
3        Azad Kashmir    PK-JK   70912109B7720427584286        PAK      ADM1   
4  Khyber Pakhtunkhwa    PK-KP  70912109B89717923239242        PAK      ADM1   

                                            geometry  
0  POLYGON ((66.68993 24.91425, 66.72033 24.91065...  
1  POLYGON ((66.68993 24.91425, 66.67194 24.87773...  
2  POLYGON ((75.49473 34.6119, 75.58785 34.6119, ...  
3  POLYGON ((74.11511 35.08527, 74.08597 35.06147...  
4  POLYGON ((70.2496 31.07038, 70.26656 31.08491,...  
Columns in GeoJSON: Index(['shapeName', 'shapeISO', 'shapeID', 'shapeGroup', 'shapeType',
       'geometry'],
      dtype='object')


In [43]:

csv_path = "data.csv"
df = pd.read_csv(csv_path)
print(df)


                      province  violence on women in 2024
0                  Balochistan                      398.0
1                        Sindh                     1781.0
2             Gilgit-Baltistan                       19.0
3                 Azad Kashmir                        NaN
4           Khyber Pakhtunkhwa                     3397.0
5  Islamabad Capital Territory                      220.0
6                       Punjab                    26753.0


In [ ]:

# the GeoJSON uses “shapeName” for province names
gdf_merged = gdf.merge(df, left_on="shapeName", right_on="province", how="left")

# 4. Create a Folium map with Choropleth
m = folium.Map(location=[30, 70], zoom_start=5)

# Use the original geojson file for geo_data
folium.Choropleth(
    geo_data=geojson_path,
    data=gdf_merged,
    columns=["shapeName", "violence on women in 2024"],
    key_on="feature.properties.shapeName",
    fill_color="YlOrRd",
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Violence on Women 2024",
    nan_fill_color="gray"
).add_to(m)


folium.GeoJson(
    gdf_merged,
    style_function=lambda feature: {
        'color': 'black', 'weight': 0.5, 'fillOpacity': 0
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["shapeName", "violence on women in 2024"],
        aliases=["Province:", "Violence against women:"]
    )
).add_to(m)

m.save("task3.html")


## Task 4: Real time API map

In [45]:

url = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson"
data = requests.get(url).json()

map4 = folium.Map(location=[20, 0], zoom_start=2)


In [50]:

marker_cluster = MarkerCluster().add_to(map4)


In [ ]:

for feature in data['features']:
    coords = feature['geometry']['coordinates']
    properties = feature['properties']
    mag = properties.get('mag', 0)
    place = properties.get('place', 'Unknown location')
    time = properties.get('time', 'No time info')

    # Color based on magnitude
    if mag < 2.5:
        color = "green"
    elif 2.5 <= mag < 5.0:
        color = "orange"
    else:
        color = "red"

    folium.CircleMarker(
        location=[coords[1], coords[0]],
        radius=max(mag * 2, 2),  
        color=color,
        fill=True,
        fill_opacity=0.7,
        popup=f"<b>Magnitude:</b> {mag}<br><b>Location:</b> {place}<br><b>Time:</b> {time}"
    ).add_to(marker_cluster)


In [52]:

map4.save("task4.html")


## References

1. Crime Data on violence against women in Pakistan in 2024: https://www.glorymagazine.pk/shocking-statistics-over-32000-cases-of-violence-against-women-reported-in-pakistan-in-2024/


2. Crime Data on Gilgit Baltistan: https://hrcp-web.org/hrcpweb/wp-content/uploads/2020/09/2024-State-of-human-rights-in-GB-in-2023-EN.pdf

